# 离线 Kaggle 推理骨架（004 第一路线，未执行、无分数）

- **离线**：最终 Kaggle notebook 关闭互联网运行；本骨架不发起任何网络请求，不读凭据。
- **输入**：自包含或显式挂载（代码/权重/候选 packed 指纹打进输入或随 notebook 打包；`kernels push` 不同步整个 src，需提前打包）。
- **动态 ID**：分子集合一律从输入 `test.parquet` 动态读取，不硬编码任何 molecule/spectrum ID，也不复用当前可见占位 IDs。
- **预算**：全量推理必须在 **<=9h** 内完成（含质量过滤 + FPNet 打分 + 聚合 + 写 submission.csv）；本骨架只计时，不承诺能跑完。
- 输出 `/kaggle/working/submission.csv`：每分子一行 25 个 SMILES（分号拼接）；空项已清洗，顺序即排名。


In [ ]:
# I0：输入与环境（动态发现，不硬编码 ID/路径）。
import os, time, json
from pathlib import Path

t_start = time.perf_counter()
INPUT_DIR = Path(os.environ.get("INPUT_DIR", "/kaggle/input"))
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
# 测试输入文件名按挂载实际发现，不假设固定数据集 slug。
cands = sorted(INPUT_DIR.rglob("test.parquet"))
print({"input_dir": str(INPUT_DIR), "test_parquet_hits": [str(p) for p in cands][:5], "n_hits": len(cands)})


In [ ]:
# I1：动态读 ID（占位演示：只读 ID 列，不做打分）。
# 真实推理在此接入：质量过滤(float64) -> packed 候选按 batch 搬 -> FPNet single/merged 路由前向 -> 候选级聚合(<=25 截断计数)。
molecule_ids = []  # 从 test.parquet 的 molecule 列动态读入；此处留空，运行时填充。
# import pyarrow.parquet as pq
# table = pq.read_table(str(cands[0]), columns=["molecule_id"])
# molecule_ids = table.column(0).to_pylist()
print({"n_molecules": len(molecule_ids), "note": "fill from test.parquet dynamically; no hardcoded IDs"})


In [ ]:
# I2：写 submission.csv（格式骨架；候选由真实排序产生，此处仅占位说明）。
# 正式行 = 25 个规范 SMILES 用 ';' 拼接；空项删除；不得输出空行或超 25 候选（官方拒绝）。
import csv
out_path = WORKING_DIR / "submission.csv"
# with open(out_path, "w", newline="", encoding="utf-8") as f:
#     w = csv.writer(f); w.writerow(["molecule_id", "smiles"])
#     for mid, ranked25 in results.items():  # results 来自真实排序
#         assert 1 <= len(ranked25) <= 25
#         w.writerow([mid, ";".join(ranked25)])
elapsed = time.perf_counter() - t_start
print({"submission": str(out_path), "elapsed_seconds": elapsed, "budget": "must stay <= 9h (32400s) end-to-end"})
